In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q rouge-score

  Preparing metadata (setup.py) ... done


In [ ]:
#evaluation dataset
import pandas as pd
import ast
import os

DRIVE_DIR = "/content/drive/MyDrive/rag-eval-metrics"
eval_path = f"{DRIVE_DIR}/ragtruth_eval_dataframe.csv"

eval_df = pd.read_csv(eval_path)

# Restore labels from their CSV string representation
eval_df["labels"] = eval_df["labels"].apply(
    lambda x: [] if pd.isna(x) or x == "" else ast.literal_eval(x)
)

print("Loaded evaluation dataframe:", eval_df.shape)
print("Columns:", eval_df.columns.tolist())
print("\nHallucination distribution:")
print(eval_df["has_hallucination"].value_counts())

Loaded evaluation dataframe: (1781, 11)
Columns: ['id', 'source_id', 'model', 'temperature', 'labels', 'split', 'quality', 'response', 'task_type', 'source_info', 'has_hallucination']

Hallucination distribution:
has_hallucination
False    1017
True      764
Name: count, dtype: int64


In [ ]:
#calculating Rouge Scores
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)

def score_row(row):
    scores = scorer.score(
        target=row["source_info"],
        prediction=row["response"]
    )

    return pd.Series({
        "rouge1_precision": scores["rouge1"].precision,
        "rouge1_recall": scores["rouge1"].recall,
        "rouge1_f1": scores["rouge1"].fmeasure,

        "rouge2_precision": scores["rouge2"].precision,
        "rouge2_recall": scores["rouge2"].recall,
        "rouge2_f1": scores["rouge2"].fmeasure,

        "rougeL_precision": scores["rougeL"].precision,
        "rougeL_recall": scores["rougeL"].recall,
        "rougeL_f1": scores["rougeL"].fmeasure
    })

rouge_scores = eval_df.apply(score_row, axis=1)

results = pd.concat([
    eval_df[
        ["id", "source_id", "model", "task_type", "has_hallucination"]
    ],
    rouge_scores
], axis=1)

print("ROUGE scoring completed.")
print("Results shape:", results.shape)
display(results.head())


ROUGE scoring completed.
Results shape: (1781, 14)


,id,source_id,model,task_type,has_hallucination,rouge1_precision,rouge1_recall,rouge1_f1,rouge2_precision,rouge2_recall,rouge2_f1,rougeL_precision,rougeL_recall,rougeL_f1
0,11538,15034,gpt-4-0613,Data2txt,False,0.489051,0.293860,0.367123,0.080882,0.048458,0.060606,0.248175,0.149123,0.186301
1,10679,14884,llama-2-70b-chat,Data2txt,True,0.383929,0.154122,0.219949,0.099099,0.039568,0.056555,0.232143,0.093190,0.132992
2,7264,14081,llama-2-13b-chat,Data2txt,True,0.483660,0.215116,0.297787,0.098684,0.043732,0.060606,0.215686,0.095930,0.132797
3,9643,14702,gpt-3.5-turbo-0613,Data2txt,False,0.507463,0.320755,0.393064,0.195000,0.123028,0.150870,0.258706,0.163522,0.200385
4,6832,14004,llama-2-13b-chat,Data2txt,True,0.569620,0.196078,0.291734,0.114650,0.039301,0.058537,0.278481,0.095861,0.142626


In [ ]:
#Compare the ROUGE scores with Human Labels

print("Mean ROUGE-1 precision by hallucination label:")
display(
    results.groupby("has_hallucination")["rouge1_precision"]
    .agg(["mean", "median", "std", "count"])
)

print("\nMean ROUGE-1 precision by task type and hallucination label:")
display(
    results.groupby(
        ["task_type", "has_hallucination"]
    )["rouge1_precision"]
    .agg(["mean", "median", "std", "count"])
)

Mean ROUGE-1 precision by hallucination label:


,mean,median,std,count
has_hallucination,,,,
False,0.760604,0.809524,0.161138,1017
True,0.616918,0.606649,0.161788,764



Mean ROUGE-1 precision by task type and hallucination label:


mean    median       std  count
task_type has_hallucination                                     
Data2txt  False              0.558673  0.561046  0.093299    220
          True               0.512080  0.514451  0.099251    407
QA        False              0.781365  0.818182  0.152838    405
          True               0.661665  0.684081  0.137672    178
Summary   False              0.852483  0.861128  0.081275    392
          True               0.810798  0.823899  0.077143    179

In [ ]:
#saving to drive
out_path = f"{DRIVE_DIR}/ragtruth_rouge_scores.csv"

results.to_csv(out_path, index=False)

print("Saved ROUGE scores to:")
print(out_path)

print("File exists:", os.path.exists(out_path))

Saved ROUGE scores to:
/content/drive/MyDrive/rag-eval-metrics/ragtruth_rouge_scores.csv
File exists: True


In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

# Convert labels to binary values
y_true = results["has_hallucination"].astype(int)

# Lower ROUGE overlap is treated as a stronger hallucination signal
y_score = -results["rouge1_precision"]

print("Overall baseline performance")
print("ROC-AUC:", round(roc_auc_score(y_true, y_score), 4))
print("PR-AUC (Average Precision):", round(
    average_precision_score(y_true, y_score), 4
))

print("\nPerformance by task type")

for task in sorted(results["task_type"].unique()):
    subset = results[results["task_type"] == task]

    y_task = subset["has_hallucination"].astype(int)
    score_task = -subset["rouge1_precision"]

    if y_task.nunique() < 2:
        print(task, "- Cannot calculate: only one label present")
        continue

    print(
        task,
        "| n =", len(subset),
        "| ROC-AUC =", round(roc_auc_score(y_task, score_task), 4),
        "| PR-AUC =", round(average_precision_score(y_task, score_task), 4)
    )

Overall baseline performance
ROC-AUC: 0.7413
PR-AUC (Average Precision): 0.6422

Performance by task type
Data2txt | n = 627 | ROC-AUC = 0.6278 | PR-AUC = 0.7622
QA | n = 583 | ROC-AUC = 0.7524 | PR-AUC = 0.4848
Summary | n = 571 | ROC-AUC = 0.6596 | PR-AUC = 0.4459


In [ ]:
#calculate F1 using a selected threshold
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    confusion_matrix,
    classification_report
)

# Positive class: hallucination
y = results["has_hallucination"].astype(int).to_numpy()

# Higher score means more likely hallucination
scores = -results["rouge1_precision"].to_numpy()

# Split into 60% train, 20% validation, 20% test
indices = np.arange(len(y))

train_idx, temp_idx = train_test_split(
    indices,
    test_size=0.40,
    random_state=42,
    stratify=y
)

val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    random_state=42,
    stratify=y[temp_idx]
)

y_val = y[val_idx]
scores_val = scores[val_idx]

y_test = y[test_idx]
scores_test = scores[test_idx]

# Choose threshold on validation data by maximizing F1
precision, recall, thresholds = precision_recall_curve(
    y_val,
    scores_val
)

# The final precision/recall point has no corresponding threshold
f1_values = (
    2 * precision[:-1] * recall[:-1] /
    (precision[:-1] + recall[:-1] + 1e-12)
)

best_idx = np.argmax(f1_values)
best_threshold = thresholds[best_idx]

print("Selected threshold (validation):", round(best_threshold, 4))
print("Validation F1:", round(f1_values[best_idx], 4))

# Evaluate once on the held-out test set
y_pred_test = (scores_test >= best_threshold).astype(int)

print("\nTest-set results")
print("Accuracy:", round(accuracy_score(y_test, y_pred_test), 4))
print("Precision:", round(precision_score(y_test, y_pred_test, zero_division=0), 4))
print("Recall:", round(recall_score(y_test, y_pred_test, zero_division=0), 4))
print("F1:", round(f1_score(y_test, y_pred_test, zero_division=0), 4))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred_test))

print("\nClassification report:")
print(classification_report(
    y_test,
    y_pred_test,
    target_names=["No hallucination", "Hallucination"],
    zero_division=0
))

Selected threshold (validation): -0.8333
Validation F1: 0.6937

Test-set results
Accuracy: 0.6246
Precision: 0.5373
Recall: 0.8954
F1: 0.6716

Confusion matrix:
[[ 86 118]
 [ 16 137]]

Classification report:
                  precision    recall  f1-score   support

No hallucination       0.84      0.42      0.56       204
   Hallucination       0.54      0.90      0.67       153

        accuracy                           0.62       357
       macro avg       0.69      0.66      0.62       357
    weighted avg       0.71      0.62      0.61       357



ROUGE achieves high recall for hallucination detection, but at the cost of many false alarms.

In [ ]:
#comparing threshold choices fairly

import numpy as np
import pandas as pd
from sklearn.metrics import (
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

# Precision-recall curve from validation data only
val_precision, val_recall, val_thresholds = precision_recall_curve(
    y_val, scores_val
)

# F1 for each available threshold
val_f1 = (
    2 * val_precision[:-1] * val_recall[:-1] /
    (val_precision[:-1] + val_recall[:-1] + 1e-12)
)

# Candidate threshold strategies, all selected on validation data
candidate_thresholds = {
    "F1-maximizing": val_thresholds[np.argmax(val_f1)],
    "Precision >= 0.60": None,
    "Recall >= 0.80": None,
    "Fixed ROUGE cutoff 0.50": -0.50
}

# Among thresholds meeting precision >= 0.60, select highest recall
eligible_precision = np.where(val_precision[:-1] >= 0.60)[0]
if len(eligible_precision) > 0:
    best_precision_idx = eligible_precision[
        np.argmax(val_recall[eligible_precision])
    ]
    candidate_thresholds["Precision >= 0.60"] = val_thresholds[best_precision_idx]

# Among thresholds meeting recall >= 0.80, select highest precision
eligible_recall = np.where(val_recall[:-1] >= 0.80)[0]
if len(eligible_recall) > 0:
    best_recall_idx = eligible_recall[
        np.argmax(val_precision[eligible_recall])
    ]
    candidate_thresholds["Recall >= 0.80"] = val_thresholds[best_recall_idx]

# Evaluate all selected thresholds on the held-out test set
comparison = []

for strategy, threshold in candidate_thresholds.items():
    if threshold is None:
        continue

    y_pred = (scores_test >= threshold).astype(int)

    comparison.append({
        "strategy": strategy,
        "threshold_on_reversed_score": threshold,
        "equivalent_ROUGE_cutoff": -threshold,
        "test_accuracy": accuracy_score(y_test, y_pred),
        "test_precision": precision_score(y_test, y_pred, zero_division=0),
        "test_recall": recall_score(y_test, y_pred, zero_division=0),
        "test_f1": f1_score(y_test, y_pred, zero_division=0)
    })

comparison_df = pd.DataFrame(comparison)
display(comparison_df.round(4))

,strategy,threshold_on_reversed_score,equivalent_ROUGE_cutoff,test_accuracy,test_precision,test_recall,test_f1
0,F1-maximizing,-0.8333,0.8333,0.6246,0.5373,0.8954,0.6716
1,Precision >= 0.60,-0.7868,0.7868,0.6807,0.5899,0.8366,0.6919
2,Recall >= 0.80,-0.8034,0.8034,0.6527,0.5622,0.8562,0.6788
3,Fixed ROUGE cutoff 0.50,-0.5000,0.5000,0.6162,0.6333,0.2484,0.3568


In [ ]:
#Adding the confidence intervals - estimate uncertainty around your held-out test metrics using bootstrap confidence intervals.
#This will help you report results more responsibly, especially because the test set contains 357 responses.

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

# Use the F1-maximizing threshold selected from validation data
selected_threshold = candidate_thresholds["F1-maximizing"]

# Predictions on the untouched test set
y_pred_test = (scores_test >= selected_threshold).astype(int)

# Metric functions
metric_functions = {
    "Accuracy": lambda yt, yp, ys: accuracy_score(yt, yp),
    "Precision": lambda yt, yp, ys: precision_score(
        yt, yp, zero_division=0
    ),
    "Recall": lambda yt, yp, ys: recall_score(
        yt, yp, zero_division=0
    ),
    "F1": lambda yt, yp, ys: f1_score(
        yt, yp, zero_division=0
    ),
    "ROC-AUC": lambda yt, yp, ys: roc_auc_score(yt, ys),
    "PR-AUC": lambda yt, yp, ys: average_precision_score(yt, ys)
}

# Reproducible bootstrap
rng = np.random.default_rng(42)
n_bootstrap = 2000
n_test = len(y_test)

bootstrap_values = {name: [] for name in metric_functions}

for _ in range(n_bootstrap):
    sample_idx = rng.integers(0, n_test, size=n_test)

    yt = y_test[sample_idx]
    yp = y_pred_test[sample_idx]
    ys = scores_test[sample_idx]

    # ROC-AUC requires both classes in the resampled data
    if len(np.unique(yt)) < 2:
        continue

    for name, metric_fn in metric_functions.items():
        bootstrap_values[name].append(metric_fn(yt, yp, ys))

# Point estimates and percentile 95% confidence intervals
rows = []

for name, metric_fn in metric_functions.items():
    point_estimate = metric_fn(y_test, y_pred_test, scores_test)
    values = np.asarray(bootstrap_values[name])

    lower = np.percentile(values, 2.5)
    upper = np.percentile(values, 97.5)

    rows.append({
        "Metric": name,
        "Test estimate": point_estimate,
        "95% CI lower": lower,
        "95% CI upper": upper
    })

ci_df = pd.DataFrame(rows)
display(ci_df.round(4))

# Save the confidence interval table to Drive
ci_path = f"{DRIVE_DIR}/rouge_baseline_bootstrap_ci.csv"
ci_df.to_csv(ci_path, index=False)

print("Saved confidence intervals to:", ci_path)


,Metric,Test estimate,95% CI lower,95% CI upper
0,Accuracy,0.6246,0.5770,0.6723
1,Precision,0.5373,0.4768,0.5962
2,Recall,0.8954,0.8456,0.9437
3,F1,0.6716,0.6174,0.7213
4,ROC-AUC,0.7543,0.7061,0.8043
5,PR-AUC,0.6743,0.5999,0.7470


Saved confidence intervals to: /content/drive/MyDrive/rag-eval-metrics/rouge_baseline_bootstrap_ci.csv


ROUGE-1 context-overlap baseline: ROUGE-1 precision was computed between generated responses and retrieved source contexts. A hallucination-detection score was defined as the negative ROUGE-1 precision, with the decision threshold selected using validation data. On the held-out test set (n = 357), the model achieved ROC-AUC = 0.7543, PR-AUC = 0.6743, and F1 = 0.6716 at the validation-selected threshold. Bootstrap 95% confidence intervals were estimated using 2,000 resamples. The selected threshold yielded high recall (0.8954) but lower precision (0.5373), indicating that context overlap can provide a useful signal but also produces many false positives.

In [ ]:
import pandas as pd

DRIVE_DIR = "/content/drive/MyDrive/rag-eval-metrics"

df = pd.read_csv(
    f"{DRIVE_DIR}/ragtruth_eval_dataframe.csv"
)

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

Dataset shape: (1781, 11)
Columns: ['id', 'source_id', 'model', 'temperature', 'labels', 'split', 'quality', 'response', 'task_type', 'source_info', 'has_hallucination']


,id,source_id,model,temperature,labels,split,quality,response,task_type,source_info,has_hallucination
0,11538,15034,gpt-4-0613,0.700,[],train,good,"""The Mex Authentic"" is a popular Mexican resta...",Data2txt,"{'name': 'The Mex Authentic', 'address': '413 ...",False
1,10679,14884,llama-2-70b-chat,0.700,"[{'start': 113, 'end': 142, 'text': 'other Ita...",train,good,Lilsy Wood Fired Pizza is a locally owned rest...,Data2txt,"{'name': 'Lilsy Wood Fired Pizza', 'address': ...",True
2,7264,14081,llama-2-13b-chat,0.775,"[{'start': 325, 'end': 342, 'text': 'and reser...",test,good,Sure! Here's an objective overview of Little A...,Data2txt,"{'name': ""Little Alex's"", 'address': '1024 Coa...",True
3,9643,14702,gpt-3.5-turbo-0613,0.700,[],train,good,Lito's Take Out is a local business located at...,Data2txt,"{'name': ""Lito's Take Out"", 'address': '514 E...",False
4,6832,14004,llama-2-13b-chat,0.700,"[{'start': 228, 'end': 254, 'text': 'open for ...",train,good,Sure! Here's an objective overview of Seagrass...,Data2txt,"{'name': 'Seagrass Restaurant', 'address': '30...",True


In [ ]:
 #train_test_split

import pandas as pd

DRIVE_DIR = "/content/drive/MyDrive/rag-eval-metrics"

# Load the canonical split created earlier
split_df = pd.read_csv(
    f"{DRIVE_DIR}/ragtruth_split_assignment.csv"
)

# Confirm the expected columns
print("Split file columns:", split_df.columns.tolist())
print("Split counts:")
print(split_df["split"].value_counts())

# Merge split assignments into the evaluation dataframe
df = df.drop(columns=["split"], errors="ignore")

df = df.merge(
    split_df[["id", "split"]],
    on="id",
    how="inner",
    validate="one_to_one"
)

assert len(df) == 1781, f"Unexpected row count: {len(df)}"
assert df["split"].notna().all()
assert df["id"].is_unique

print("\nCanonical split loaded successfully.")
print(df["split"].value_counts())

Split file columns: ['id', 'split']
Split counts:
split
train    1068
test      357
val       356
Name: count, dtype: int64

Canonical split loaded successfully.
split
train    1068
test      357
val       356
Name: count, dtype: int64


In [ ]:
#Re-run threshold evaluation and confidence intervals

import numpy as np
import pandas as pd

from sklearn.metrics import (
    precision_recall_curve,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

# Load the saved canonical split
split_path = f"{DRIVE_DIR}/ragtruth_split_assignment.csv"
split_df = pd.read_csv(split_path)

# Attach split membership to the ROUGE results
canonical_results = results.merge(
    split_df,
    on="id",
    how="inner",
    validate="one_to_one"
)

assert len(canonical_results) == len(results) == 1781
assert canonical_results["split"].notna().all()

# Hallucination labels and reversed ROUGE-1 precision
# Higher score = stronger hallucination signal
canonical_results["y_true"] = (
    canonical_results["has_hallucination"].astype(int)
)
canonical_results["hallucination_score"] = (
    -canonical_results["rouge1_precision"]
)

# Separate validation and test sets
val_df = canonical_results[canonical_results["split"] == "val"].copy()
test_df = canonical_results[canonical_results["split"] == "test"].copy()

y_val = val_df["y_true"].to_numpy()
scores_val = val_df["hallucination_score"].to_numpy()

y_test = test_df["y_true"].to_numpy()
scores_test = test_df["hallucination_score"].to_numpy()

# Select threshold using validation data only
precision_val, recall_val, thresholds_val = precision_recall_curve(
    y_val, scores_val
)

f1_val = (
    2 * precision_val[:-1] * recall_val[:-1]
    / (precision_val[:-1] + recall_val[:-1] + 1e-12)
)

best_idx = np.argmax(f1_val)
best_threshold = thresholds_val[best_idx]

print("Canonical split sizes:")
print(canonical_results["split"].value_counts())

print("\nSelected threshold (validation):", round(best_threshold, 4))
print("Validation F1:", round(f1_val[best_idx], 4))

# Evaluate on held-out test data
y_pred_test = (scores_test >= best_threshold).astype(int)

test_metrics = {
    "Accuracy": accuracy_score(y_test, y_pred_test),
    "Precision": precision_score(y_test, y_pred_test, zero_division=0),
    "Recall": recall_score(y_test, y_pred_test, zero_division=0),
    "F1": f1_score(y_test, y_pred_test, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, scores_test),
    "PR-AUC": average_precision_score(y_test, scores_test)
}

print("\nCanonical test-set results:")
for metric, value in test_metrics.items():
    print(f"{metric}: {value:.4f}")

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred_test))

# Bootstrap 95% confidence intervals on the test set
rng = np.random.default_rng(42)
n_bootstrap = 2000
n_test = len(y_test)

bootstrap_values = {metric: [] for metric in test_metrics}

for _ in range(n_bootstrap):
    sample_idx = rng.integers(0, n_test, size=n_test)

    yt = y_test[sample_idx]
    yp = y_pred_test[sample_idx]
    ys = scores_test[sample_idx]

    # ROC-AUC and PR-AUC require both classes in the resample
    if len(np.unique(yt)) < 2:
        continue

    bootstrap_values["Accuracy"].append(accuracy_score(yt, yp))
    bootstrap_values["Precision"].append(
        precision_score(yt, yp, zero_division=0)
    )
    bootstrap_values["Recall"].append(
        recall_score(yt, yp, zero_division=0)
    )
    bootstrap_values["F1"].append(f1_score(yt, yp, zero_division=0))
    bootstrap_values["ROC-AUC"].append(roc_auc_score(yt, ys))
    bootstrap_values["PR-AUC"].append(average_precision_score(yt, ys))

ci_rows = []

for metric, estimate in test_metrics.items():
    values = np.asarray(bootstrap_values[metric])

    ci_rows.append({
        "Metric": metric,
        "Test estimate": estimate,
        "95% CI lower": np.percentile(values, 2.5),
        "95% CI upper": np.percentile(values, 97.5)
    })

canonical_ci_df = pd.DataFrame(ci_rows)

print("\nBootstrap 95% confidence intervals:")
display(canonical_ci_df.round(4))

# Save canonical results
canonical_ci_path = f"{DRIVE_DIR}/rouge_canonical_test_bootstrap_ci.csv"
canonical_ci_df.to_csv(canonical_ci_path, index=False)

print("\nSaved canonical confidence intervals to:", canonical_ci_path)

Canonical split sizes:
split
train    1068
test      357
val       356
Name: count, dtype: int64

Selected threshold (validation): -0.8118
Validation F1: 0.7016

Canonical test-set results:
Accuracy: 0.6527
Precision: 0.5622
Recall: 0.8562
F1: 0.6788
ROC-AUC: 0.7298
PR-AUC: 0.6060

Confusion matrix:
[[102 102]
 [ 22 131]]

Bootstrap 95% confidence intervals:


,Metric,Test estimate,95% CI lower,95% CI upper
0,Accuracy,0.6527,0.6050,0.7031
1,Precision,0.5622,0.4979,0.6227
2,Recall,0.8562,0.7987,0.9110
3,F1,0.6788,0.6218,0.7299
4,ROC-AUC,0.7298,0.6769,0.7798
5,PR-AUC,0.6060,0.5260,0.6975



Saved canonical confidence intervals to: /content/drive/MyDrive/rag-eval-metrics/rouge_canonical_test_bootstrap_ci.csv


BERT SCORE

In [ ]:
#Install BERTScore and import the required libraries
%pip install -q bert-score

import pandas as pd
import numpy as np
import torch
from bert_score import score

print("BERTScore installed and imported successfully.")
print("GPU available:", torch.cuda.is_available())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.4 MB/s eta 0:00:00
BERTScore installed and imported successfully.
GPU available: True


In [ ]:
#Check the data needed for BERTScore
# Confirm required columns are present
required_columns = ["id", "response", "source_info", "has_hallucination", "split"]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

print("Missing columns:", missing_columns)
print("Dataset shape:", df.shape)
print("\nSplit counts:")
print(df["split"].value_counts())

# Check for missing text or labels
print("\nMissing responses:", df["response"].isna().sum())
print("Missing contexts:", df["source_info"].isna().sum())
print("Missing hallucination labels:", df["has_hallucination"].isna().sum())

df[required_columns].head()

Missing columns: []
Dataset shape: (1781, 11)

Split counts:
split
train    1068
test      357
val       356
Name: count, dtype: int64

Missing responses: 0
Missing contexts: 0
Missing hallucination labels: 0


,id,response,source_info,has_hallucination,split
0,11538,"""The Mex Authentic"" is a popular Mexican resta...","{'name': 'The Mex Authentic', 'address': '413 ...",False,train
1,10679,Lilsy Wood Fired Pizza is a locally owned rest...,"{'name': 'Lilsy Wood Fired Pizza', 'address': ...",True,train
2,7264,Sure! Here's an objective overview of Little A...,"{'name': ""Little Alex's"", 'address': '1024 Coa...",True,test
3,9643,Lito's Take Out is a local business located at...,"{'name': ""Lito's Take Out"", 'address': '514 E...",False,test
4,6832,Sure! Here's an objective overview of Seagrass...,"{'name': 'Seagrass Restaurant', 'address': '30...",True,train


In [ ]:
#Check BERTScore truncation risk
# Use the dataframe already loaded and merged with the canonical split
eval_df = df.copy()

# Rough truncation-risk check (word count, not exact token count)
approx_limit_words = 400  # Approximate proxy for ~512 English tokens

n_over = (
    eval_df["source_info"]
    .astype(str)
    .str.split()
    .str.len()
    > approx_limit_words
).sum()

print(
    f"Rows with source_info longer than ~{approx_limit_words} words "
    f"(truncation risk): {n_over} of {len(eval_df)} "
    f"({100 * n_over / len(eval_df):.1f}%)"
)

Rows with source_info longer than ~400 words (truncation risk): 495 of 1781 (27.8%)


In [ ]:
#Compute BERT Score
from bert_score import score as bertscore_score

P, R, F1 = bertscore_score(
    cands=eval_df['response'].tolist(),
    refs=eval_df['source_info'].tolist(),
    lang='en',
    model_type='roberta-large',
    verbose=True,
    device='cuda' if torch.cuda.is_available() else 'cpu',
)

bert_results = eval_df[
    ['id', 'source_id', 'model', 'task_type', 'has_hallucination']
].copy()

bert_results['bertscore_precision'] = P.numpy()
bert_results['bertscore_recall'] = R.numpy()
bert_results['bertscore_f1'] = F1.numpy()

print(bert_results.head())

bert_out_path = f'{DRIVE_DIR}/ragtruth_bertscore_scores.csv'
bert_results.to_csv(bert_out_path, index=False)

print('\nSaved per-response BERTScore results to', bert_out_path)

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/50 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/28 [00:00<?, ?it/s]

done in 169.47 seconds, 10.51 sentences/sec
      id  source_id               model task_type  has_hallucination  \
0  11538      15034          gpt-4-0613  Data2txt              False   
1  10679      14884    llama-2-70b-chat  Data2txt               True   
2   7264      14081    llama-2-13b-chat  Data2txt               True   
3   9643      14702  gpt-3.5-turbo-0613  Data2txt              False   
4   6832      14004    llama-2-13b-chat  Data2txt               True   

   bertscore_precision  bertscore_recall  bertscore_f1  
0             0.855291          0.771759      0.811381  
1             0.842040          0.748122      0.792308  
2             0.826565          0.757526      0.790541  
3             0.835493          0.760782      0.796389  
4             0.834594          0.765361      0.798480  

Saved per-response BERTScore results to /content/drive/MyDrive/rag-eval-metrics/ragtruth_bertscore_scores.csv


In [ ]:
#Grouped BERTScore results
# Grouped mean BERTScore by hallucination label
print("=== BERTScore by Hallucination Label ===")

bert_by_label = (
    bert_results
    .groupby("has_hallucination")[
        ["bertscore_precision", "bertscore_recall", "bertscore_f1"]
    ]
    .mean()
    .round(4)
)

display(bert_by_label)


# Grouped mean BERTScore by task type and hallucination label
print("\n=== BERTScore by Task Type and Hallucination Label ===")

bert_by_task_label = (
    bert_results
    .groupby(["task_type", "has_hallucination"])[
        ["bertscore_precision", "bertscore_recall", "bertscore_f1"]
    ]
    .mean()
    .round(4)
)

display(bert_by_task_label)

=== BERTScore by Hallucination Label ===


,bertscore_precision,bertscore_recall,bertscore_f1
has_hallucination,,,
False,0.8889,0.8232,0.8547
True,0.8624,0.7979,0.8287



=== BERTScore by Task Type and Hallucination Label ===


bertscore_precision  bertscore_recall  \
task_type has_hallucination                                          
Data2txt  False                           0.8469            0.7698   
          True                            0.8404            0.7621   
QA        False                           0.8985            0.8392   
          True                            0.8802            0.8460   
Summary   False                           0.9026            0.8367   
          True                            0.8947            0.8315   

                             bertscore_f1  
task_type has_hallucination                
Data2txt  False                    0.8065  
          True                     0.7993  
QA        False                    0.8677  
          True                     0.8627  
Summary   False                    0.8683  
          True                     0.8619

In [ ]:
#Select the threshold on validation and evaluate on test
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
)

# Attach the already-existing canonical split to BERTScore results
split_lookup = eval_df[["id", "split"]].drop_duplicates()

bert_eval = bert_results.merge(
    split_lookup,
    on="id",
    how="left",
    validate="one_to_one"
)

assert bert_eval["split"].notna().all()
assert len(bert_eval) == 1781

# Lower BERTScore precision = more likely hallucination
val_df = bert_eval[bert_eval["split"] == "val"].copy()
test_df = bert_eval[bert_eval["split"] == "test"].copy()

y_val = val_df["has_hallucination"].astype(int).to_numpy()
val_scores = val_df["bertscore_precision"].to_numpy()

y_test = test_df["has_hallucination"].astype(int).to_numpy()
test_scores = test_df["bertscore_precision"].to_numpy()

# Choose threshold using validation data only
candidate_thresholds = np.sort(np.unique(val_scores))

best_threshold = None
best_val_f1 = -1

for threshold in candidate_thresholds:
    val_predictions = (val_scores <= threshold).astype(int)
    current_f1 = f1_score(y_val, val_predictions, zero_division=0)

    if current_f1 > best_val_f1:
        best_val_f1 = current_f1
        best_threshold = threshold

print(f"Selected validation threshold: {best_threshold:.6f}")
print(f"Validation F1: {best_val_f1:.4f}")

# Apply the fixed threshold to the held-out test set
test_predictions = (test_scores <= best_threshold).astype(int)

test_metrics = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC-AUC",
        "PR-AUC",
    ],
    "Score": [
        accuracy_score(y_test, test_predictions),
        precision_score(y_test, test_predictions, zero_division=0),
        recall_score(y_test, test_predictions, zero_division=0),
        f1_score(y_test, test_predictions, zero_division=0),
        roc_auc_score(y_test, -test_scores),
        average_precision_score(y_test, -test_scores),
    ],
})

print("\n=== BERTScore Held-out Test Metrics ===")
display(test_metrics.round(4))

print("\nConfusion matrix [[TN, FP], [FN, TP]]:")
print(confusion_matrix(y_test, test_predictions))

Selected validation threshold: 0.902768
Validation F1: 0.6815

=== BERTScore Held-out Test Metrics ===


,Metric,Score
0,Accuracy,0.5770
1,Precision,0.5036
2,Recall,0.9020
3,F1,0.6464
4,ROC-AUC,0.7219
5,PR-AUC,0.6095



Confusion matrix [[TN, FP], [FN, TP]]:
[[ 68 136]
 [ 15 138]]


In [ ]:
#Bootstrap 95% confidence intervals
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
)

# Fixed threshold selected using the validation set
threshold = best_threshold

# Test-set predictions
test_predictions = (test_scores <= threshold).astype(int)

# Metric calculation
def calculate_metrics(y_true, predictions, scores):
    metrics = {
        "Accuracy": accuracy_score(y_true, predictions),
        "Precision": precision_score(
            y_true, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_true, predictions, zero_division=0
        ),
        "F1": f1_score(
            y_true, predictions, zero_division=0
        ),
    }

    # AUC metrics require both classes to be present
    if len(np.unique(y_true)) == 2:
        metrics["ROC-AUC"] = roc_auc_score(y_true, -scores)
        metrics["PR-AUC"] = average_precision_score(y_true, -scores)
    else:
        metrics["ROC-AUC"] = np.nan
        metrics["PR-AUC"] = np.nan

    return metrics


# Point estimates from the original test set
point_estimates = calculate_metrics(
    y_test,
    test_predictions,
    test_scores
)

# Bootstrap resampling
n_bootstrap = 2000
rng = np.random.default_rng(42)

bootstrap_results = {
    metric: [] for metric in point_estimates
}

n_test = len(y_test)

for _ in range(n_bootstrap):
    indices = rng.integers(0, n_test, size=n_test)

    y_boot = y_test[indices]
    pred_boot = test_predictions[indices]
    score_boot = test_scores[indices]

    boot_metrics = calculate_metrics(
        y_boot,
        pred_boot,
        score_boot
    )

    for metric, value in boot_metrics.items():
        bootstrap_results[metric].append(value)

# Calculate percentile 95% confidence intervals
ci_rows = []

for metric, estimate in point_estimates.items():
    values = np.array(bootstrap_results[metric], dtype=float)
    values = values[~np.isnan(values)]

    lower = np.percentile(values, 2.5)
    upper = np.percentile(values, 97.5)

    ci_rows.append({
        "Metric": metric,
        "Estimate": estimate,
        "CI_95_Lower": lower,
        "CI_95_Upper": upper,
    })

bert_bootstrap_ci = pd.DataFrame(ci_rows).round(4)

print("=== BERTScore Test Metrics with 95% Bootstrap CIs ===")
display(bert_bootstrap_ci)

# Save results to Google Drive
ci_path = f"{DRIVE_DIR}/bertscore_canonical_test_bootstrap_ci.csv"
bert_bootstrap_ci.to_csv(ci_path, index=False)

print("\nSaved bootstrap confidence intervals to:", ci_path)


=== BERTScore Test Metrics with 95% Bootstrap CIs ===


,Metric,Estimate,CI_95_Lower,CI_95_Upper
0,Accuracy,0.5770,0.5238,0.6246
1,Precision,0.5036,0.4453,0.5600
2,Recall,0.9020,0.8481,0.9434
3,F1,0.6464,0.5917,0.6953
4,ROC-AUC,0.7219,0.6676,0.7724
5,PR-AUC,0.6095,0.5263,0.7009



Saved bootstrap confidence intervals to: /content/drive/MyDrive/rag-eval-metrics/bertscore_canonical_test_bootstrap_ci.csv


this cell is to estimate uncertainty around the held-out test metrics using 2,000 bootstrap resamples. It keeps the validation-selected threshold fixed; it does not tune the threshold on the test set.

In [ ]:
import pandas as pd

metadata_path = '/content/drive/MyDrive/rag-eval-metrics/metric_run_metadata.csv'

rows = [
    {
        'metric': 'rouge1_precision', 'model_or_method': 'rouge-score (ROUGE-1)',
        'device': 'cpu', 'compute_time_seconds': None,
        'val_threshold': 0.8118, 'val_f1': 0.7016,
        'test_accuracy': 0.6527, 'test_precision': 0.5622, 'test_recall': 0.8562,
        'test_f1': 0.6788, 'test_roc_auc': 0.7298, 'test_pr_auc': 0.6060,
    },
    {
        'metric': 'bertscore_precision', 'model_or_method': 'roberta-large',
        'device': 'cuda (T4)', 'compute_time_seconds': 169.47,
        'val_threshold': 0.902768, 'val_f1': 0.6815,
        'test_accuracy': 0.5770, 'test_precision': 0.5036, 'test_recall': 0.9020,
        'test_f1': 0.6464, 'test_roc_auc': 0.7219, 'test_pr_auc': 0.6095,
    },
]

pd.DataFrame(rows).to_csv(metadata_path, index=False)
print('Saved metadata for', len(rows), 'metrics to', metadata_path)

Saved metadata for 2 metrics to /content/drive/MyDrive/rag-eval-metrics/metric_run_metadata.csv
